# Instalación y administración de servicios de nombres de dominio (DNS)

**La columna vertebral de Internet y de las redes locales**  
Profesor: **Enrique Sainz-Terrones**

Este cuaderno convierte y reorganiza el contenido de la presentación original sobre DNS en un formato de trabajo para Jupyter Notebook.

## Objetivos del cuaderno

Al finalizar, deberías ser capaz de:

- Explicar qué es el DNS y por qué es necesario.
- Describir la estructura jerárquica del sistema de nombres de dominio.
- Comprender el proceso de resolución de nombres paso a paso.
- Diferenciar entre zonas primarias, zonas secundarias y transferencias de zona.
- Reconocer los principales tipos de registros DNS.
- Entender el funcionamiento del DNS dinámico, los reenviadores y la resolución inversa.
- Utilizar comandos básicos de diagnóstico DNS.


## 1. ¿Qué es el DNS?

El **Sistema de Nombres de Dominio** o **DNS** (*Domain Name System*) es el sistema que traduce nombres de dominio legibles para las personas, como:

```text
www.ejemplo.com
```

en direcciones IP que los dispositivos utilizan para comunicarse en una red, por ejemplo:

```text
192.0.2.1
```

Dicho de otra forma, el DNS actúa como una especie de **directorio telefónico de Internet**: permite acceder a servicios mediante nombres fáciles de recordar en lugar de tener que memorizar direcciones numéricas.


### Importancia del DNS

El DNS es esencial porque:

- Permite navegar por Internet usando nombres de dominio en lugar de direcciones IP.
- Facilita la organización jerárquica de nombres.
- Permite crear dominios específicos para empresas, instituciones y organizaciones.
- Es necesario para muchos servicios de red, como páginas web, correo electrónico, aplicaciones en la nube o servicios internos de una empresa.

### Ejemplos de uso

Algunos usos habituales del DNS son:

- Acceder a sitios web.
- Enviar correos electrónicos mediante registros **MX**.
- Localizar servicios en la nube y aplicaciones web.
- Acceder a recursos internos de una red local mediante nombres fáciles de recordar.


> **Actividad rápida**  
> Piensa en tres servicios que utilices a diario y que dependan de DNS. Por ejemplo, una web, una plataforma educativa o un servicio de correo.


## 2. DNS como sistema jerárquico y descentralizado

El DNS es un sistema **jerárquico** y **descentralizado**.

- **Jerárquico**, porque está organizado en niveles.
- **Descentralizado**, porque no existe un único servidor que almacene toda la información DNS mundial.

Un nombre de dominio completo se puede entender como una serie de niveles separados por puntos:

```text
www.empresa.com.
```

El punto final representa la **raíz** del sistema DNS. Normalmente no se escribe en el navegador, pero está implícito.


### Niveles principales del DNS

| Nivel | Ejemplo | Función |
|---|---|---|
| Raíz | `.` | Nivel más alto de la jerarquía DNS. Dirige hacia los TLD. |
| Dominio de primer nivel, TLD | `.com`, `.org`, `.es` | Agrupa dominios bajo una categoría o territorio. |
| Dominio de segundo nivel | `empresa` en `empresa.com` | Normalmente registrado por una organización o persona. |
| Subdominios | `ventas` en `ventas.empresa.com` | Permiten organizar servicios o áreas dentro de un dominio. |
| Host o servicio | `www`, `mail`, `ftp` | Identifica un equipo o servicio concreto. |

Ejemplo:

```text
ventas.empresa.com.
│      │       │   │
│      │       │   └── Raíz
│      │       └────── TLD
│      └────────────── Dominio de segundo nivel
└───────────────────── Subdominio
```


### Servidores raíz

Los servidores raíz son el nivel más alto de la jerarquía DNS. No conocen directamente la dirección IP de todos los dominios, pero sí saben a qué servidores hay que consultar para cada dominio de primer nivel, como `.com`, `.org` o `.es`.

En la práctica, cuando un servidor DNS recursivo no sabe resolver un dominio, puede empezar preguntando a los servidores raíz.


## 3. Ejemplo de resolución de nombres

Imaginemos que escribimos en el navegador la siguiente URL:

```text
http://iesalonsodemadrigal.centros.educa.jcyl.es/sitio/
```

Aunque la URL incluye un protocolo (`http://`) y una ruta (`/sitio/`), el DNS solo se encarga de resolver el **nombre de dominio**:

```text
iesalonsodemadrigal.centros.educa.jcyl.es
```

Es decir, el DNS transforma ese nombre en una dirección IP. La ruta `/sitio/` será gestionada después por el servidor web, no por DNS.


### Paso 1. Consulta del navegador

Cuando escribimos la URL, el navegador extrae el nombre de dominio y necesita conocer su dirección IP para poder conectarse al servidor correspondiente.

```text
iesalonsodemadrigal.centros.educa.jcyl.es → ¿qué dirección IP tiene?
```


### Paso 2. Consulta de caché local

Antes de preguntar a otros servidores, el sistema operativo comprueba si ya tiene guardada la respuesta en su **caché DNS local**.

- Si la IP está en caché, se usa directamente.
- Si no está en caché, se realiza una consulta DNS.


### Paso 3. Consulta al servidor DNS recursivo

El equipo envía la consulta a su **servidor DNS recursivo**.

Este servidor suele estar configurado por:

- El proveedor de Internet.
- La red de la empresa o centro educativo.
- El propio usuario, por ejemplo usando servidores públicos como `8.8.8.8`.

El servidor recursivo intenta obtener la respuesta consultando a otros servidores DNS si no la tiene ya en caché.


### Paso 4. Consulta al servidor raíz

Si el servidor recursivo no conoce la respuesta, consulta a uno de los servidores raíz.

El servidor raíz no conoce la IP final de:

```text
iesalonsodemadrigal.centros.educa.jcyl.es
```

pero sí puede indicar qué servidores gestionan el TLD `.es`.


### Paso 5. Consulta al servidor TLD `.es`

El servidor DNS recursivo consulta ahora a un servidor del TLD `.es`.

Este servidor tampoco conoce necesariamente la IP final, pero puede indicar qué servidores gestionan el dominio `jcyl.es`.


### Paso 6. Consulta al servidor autoritativo de `jcyl.es`

El servidor recursivo consulta al servidor DNS autoritativo de `jcyl.es`.

Un **servidor DNS autoritativo** es el servidor que tiene autoridad para responder de forma oficial sobre los nombres de una zona o dominio concreto.

Este servidor puede indicar cómo continuar la resolución hacia `educa.jcyl.es`.


### Paso 7. Consulta al servidor autoritativo de `educa.jcyl.es`

El servidor DNS recursivo consulta ahora al servidor que gestiona el subdominio:

```text
educa.jcyl.es
```

Este servidor responde con la información necesaria para llegar al dominio completo:

```text
iesalonsodemadrigal.centros.educa.jcyl.es
```


### Paso 8. Respuesta al navegador

Cuando el servidor DNS recursivo obtiene la dirección IP, se la devuelve al equipo cliente.

El navegador ya puede usar esa IP para iniciar la conexión con el servidor web.


### Paso 9. Establecimiento de la conexión

El navegador se conecta a la dirección IP obtenida y solicita el recurso correspondiente.

En este punto, la ruta de la URL, por ejemplo:

```text
/sitio/
```

ya no depende de DNS, sino del servidor web.


### Resumen del proceso de resolución

```text
Cliente / Navegador
      │
      ▼
Caché local
      │
      ▼
Servidor DNS recursivo
      │
      ▼
Servidor raíz
      │
      ▼
Servidor TLD .es
      │
      ▼
Servidor autoritativo de jcyl.es
      │
      ▼
Servidor autoritativo de educa.jcyl.es
      │
      ▼
Respuesta con la IP del dominio solicitado
```

La resolución DNS se basa en una cadena de consultas en la que cada servidor aporta la información necesaria para avanzar al siguiente nivel.


> **Comprueba que lo entiendes**  
> ¿Por qué el servidor raíz no necesita conocer la IP de todos los dominios existentes?  
> ¿Qué ventaja tiene dividir la resolución en niveles?


## 4. Servidores autoritativos

Los servidores autoritativos son los que una organización configura para responder oficialmente por su propio dominio.

Por ejemplo, si una empresa registra:

```text
empresa.com
```

puede configurar servidores autoritativos como:

```text
ns1.empresa.com
ns2.empresa.com
```

Estos servidores pueden estar:

- En el proveedor de dominio o hosting.
- En servicios como Cloudflare, Google Cloud DNS, OVH, etc.
- En un servidor propio, como Bind9 en Linux o DNS Server en Windows Server.
- En varias ubicaciones para mejorar redundancia y disponibilidad.


### Ejemplo de servidores autoritativos

Supongamos que se registra el dominio:

```text
ies-alonso.edu.es
```

y se definen estos servidores:

```text
ns1.ies-alonso.edu.es      → servidor ubicado físicamente en el instituto
ns2.cloud-provider.net     → servidor ubicado en un centro de datos externo
```

Ambos pueden ser autoritativos para la zona `ies-alonso.edu.es`, aunque estén en ubicaciones diferentes.


## 5. Zonas DNS

Una **zona DNS** es una parte del espacio de nombres de dominio que se administra de forma independiente por un servidor DNS o por un conjunto de servidores.

Por ejemplo, si el dominio principal es:

```text
ejemplo.com
```

la zona `ejemplo.com` podría contener registros como:

```text
www.ejemplo.com
mail.ejemplo.com
ftp.ejemplo.com
```

El servidor que gestiona la zona almacena los registros necesarios para resolver los nombres incluidos en ella.


### Ventajas de usar zonas DNS

Las zonas DNS permiten:

- **Delegación:** dividir la administración de partes del espacio de nombres.
- **Escalabilidad:** gestionar grandes dominios de forma organizada.
- **Redundancia:** usar zonas secundarias para mejorar disponibilidad.
- **Organización:** separar servicios, subdominios y responsabilidades.


### Tipos de servidores de zona

| Tipo de servidor | Descripción |
|---|---|
| Servidor de zona primaria o maestro | Tiene la copia original y editable de los registros de la zona. |
| Servidor de zona secundaria o esclavo | Mantiene una copia de la zona obtenida desde el servidor primario. |

El servidor secundario mejora la disponibilidad, ya que puede responder consultas si el servidor primario falla.


## 6. Transferencias de zona

Una **transferencia de zona** es el proceso mediante el cual un servidor DNS secundario copia los registros DNS desde un servidor primario para mantener su información actualizada.

### Tipos principales

| Tipo | Significado | Descripción |
|---|---|---|
| AXFR | Transferencia completa | Copia todos los registros de la zona. Suele usarse en la primera sincronización. |
| IXFR | Transferencia incremental | Copia solo los cambios realizados desde la última transferencia. |


### Proceso de una transferencia de zona

1. El servidor secundario solicita una transferencia de zona al servidor primario.
2. El servidor primario comprueba si el secundario tiene permiso.
3. Si la transferencia está autorizada, el primario envía los datos.
4. El servidor secundario actualiza su copia de la zona.

Este proceso permite que la información DNS sea coherente entre servidores primarios y secundarios.


### Importancia de las transferencias de zona

Las transferencias de zona son importantes porque proporcionan:

- **Redundancia y disponibilidad:** si un servidor falla, otro puede responder.
- **Balanceo de carga:** varias máquinas pueden responder a las consultas.
- **Coherencia:** los servidores secundarios mantienen una copia actualizada.

> **Nota de seguridad**  
> Las transferencias de zona deben limitarse a servidores autorizados. Una transferencia AXFR abierta puede revelar información sensible sobre la estructura interna de un dominio.


## 7. Tipos de registros DNS

Un **registro DNS** es una entrada en la base de datos de una zona DNS. Los registros indican cómo se resuelven los nombres y qué servicios están asociados a un dominio.


### Registros más habituales

| Registro | Nombre | Función |
|---|---|---|
| A | Address Record | Asocia un nombre de dominio con una dirección IPv4. |
| AAAA | Quad-A Record | Asocia un nombre de dominio con una dirección IPv6. |
| MX | Mail Exchanger | Indica los servidores de correo responsables de recibir mensajes para el dominio. |
| CNAME | Canonical Name | Crea un alias de un nombre hacia otro nombre canónico. |
| TXT | Text Record | Almacena texto asociado al dominio, por ejemplo SPF, DKIM o verificaciones. |
| NS | Name Server | Indica qué servidores DNS son autoritativos para una zona. |
| PTR | Pointer Record | Permite resolución inversa: de IP a nombre. |


### Ejemplos de registros

```text
www.ejemplo.com.      IN  A      192.0.2.10
ejemplo.com.          IN  MX 10  mail.ejemplo.com.
blog.ejemplo.com.     IN  CNAME  www.ejemplo.com.
ejemplo.com.          IN  TXT    "v=spf1 mx -all"
ejemplo.com.          IN  NS     ns1.ejemplo.com.
```

En el caso de los registros MX, la prioridad permite establecer el orden de preferencia de los servidores de correo.


> **Actividad rápida**  
> Indica qué tipo de registro DNS usarías en cada caso:
>
> 1. Asociar `www.empresa.local` con la IP `192.168.1.20`.
> 2. Indicar el servidor de correo de `empresa.local`.
> 3. Crear un alias `blog.empresa.local` hacia `www.empresa.local`.
> 4. Asociar una IP con un nombre para resolución inversa.


## 8. DNS dinámico (DDNS)

Una **IP dinámica** es una dirección IP que puede cambiar con el tiempo, normalmente porque la asigna un servidor DHCP o el proveedor de Internet.

Esto puede ser un problema si queremos alojar un servicio en una red con IP pública cambiante, porque el registro DNS podría quedar desactualizado.

El **DNS dinámico** o **DDNS** permite asociar un nombre de dominio a una dirección IP dinámica, actualizando automáticamente el registro DNS cuando cambia la IP.


### Funcionamiento básico de DDNS

1. El servidor o router obtiene una IP dinámica.
2. Un cliente DDNS detecta si la IP pública ha cambiado.
3. El cliente envía la actualización al proveedor DDNS.
4. El proveedor actualiza el registro DNS.
5. Los usuarios siguen accediendo mediante el mismo nombre de dominio.

```text
mi-servidor.ddns.net → IP pública actualizada automáticamente
```


### Escenarios comunes de uso

El DDNS se utiliza habitualmente para:

- Publicar un servidor web o FTP en una conexión doméstica o pequeña oficina.
- Acceder remotamente a una red.
- Conectarse a cámaras IP o dispositivos IoT.
- Evitar contratar una IP fija cuando no es imprescindible.

### Proveedores habituales

Algunos proveedores de DNS dinámico son:

- No-IP.
- DuckDNS.
- DynDNS.
- Cloudflare mediante scripts de actualización.


### Limitaciones del DDNS

Aunque el DDNS es útil, tiene algunas limitaciones:

- Depende del proveedor DDNS.
- Requiere un cliente instalado en el router o servidor.
- Puede existir un pequeño retraso entre el cambio de IP y la actualización DNS.
- No sustituye a una buena configuración de seguridad si se publican servicios hacia Internet.


## 9. Reenviadores DNS

Un **reenviador DNS** es un servidor al que otro servidor DNS envía las consultas que no puede resolver por sí mismo.

Por ejemplo, un servidor DNS local puede resolver nombres internos de una empresa y reenviar las consultas de dominios externos a un DNS público como:

```text
8.8.8.8
1.1.1.1
```


### Tipos de reenviadores

| Tipo | Descripción | Ejemplo |
|---|---|---|
| Reenviador directo | Envía todas las consultas externas a otro servidor DNS. | Enviar consultas de Internet a `8.8.8.8`. |
| Reenviador condicional | Reenvía consultas según el dominio solicitado. | Enviar `empresa.local` a un DNS interno específico. |

Los reenviadores ayudan a optimizar el tráfico, centralizar la resolución y reducir la carga de los servidores locales.


## 10. Resolución inversa

La **resolución inversa** es el proceso contrario a la resolución directa.

- Resolución directa: nombre → IP.
- Resolución inversa: IP → nombre.

Para ello se utilizan registros **PTR** (*Pointer Records*).

Ejemplo:

```text
192.168.1.20 → servidor1.empresa.local
```


### ¿Cuándo se usa la resolución inversa?

La resolución inversa se usa en situaciones como:

- Verificación de servidores de correo.
- Diagnóstico de red con herramientas como `traceroute`, `ping` o `nslookup`.
- Identificación de máquinas en una red.
- Tareas de seguridad y auditoría.


## 11. Comandos útiles para diagnosticar DNS

En esta parte del cuaderno se recogen algunos comandos útiles para comprobar y diagnosticar el funcionamiento de DNS.

> **Importante**  
> Algunas celdas usan comandos de sistema. Pueden requerir Linux, WSL, macOS o tener instaladas herramientas como `dnsutils`. En Windows, `nslookup` está disponible normalmente desde CMD o PowerShell.


### 11.1. `nslookup`

`nslookup` permite consultar servidores DNS y obtener información sobre nombres de dominio o direcciones IP.

Ejemplos:

```bash
nslookup www.ejemplo.com
nslookup 8.8.8.8
nslookup www.ejemplo.com 8.8.8.8
```


In [ ]:
# Consulta DNS usando nslookup desde Jupyter.
# Esta celda puede funcionar si nslookup está disponible en el sistema.
!nslookup www.ejemplo.com


### 11.2. `dig`

`dig` (*Domain Information Groper*) es una herramienta muy potente para diagnósticos DNS avanzados.

Ejemplos:

```bash
dig www.ejemplo.com
dig www.ejemplo.com +short
dig www.ejemplo.com +trace
dig @8.8.8.8 www.ejemplo.com
```


In [ ]:
# Consulta DNS usando dig.
# En Ubuntu/Debian, si no está instalado, suele estar en el paquete dnsutils.
!dig www.ejemplo.com +short


### 11.3. `host`

`host` es un comando sencillo para realizar consultas directas e inversas.

Ejemplos:

```bash
host www.ejemplo.com
host 8.8.8.8
```


In [ ]:
# Consulta DNS usando host.
!host www.ejemplo.com


### 11.4. `ping`

`ping` no es una herramienta específica de DNS, pero al hacer ping a un dominio se comprueba si el sistema puede resolver su nombre a una dirección IP.

Ejemplo:

```bash
ping www.ejemplo.com
```

Si el nombre se resuelve correctamente, veremos la IP de destino al iniciar la prueba.


In [ ]:
# Ejemplo con ping.
# En algunos sistemas conviene limitar el número de paquetes con -c 4.
!ping -c 4 www.ejemplo.com


## 12. Pruebas DNS con Python

También podemos hacer algunas pruebas sencillas desde Python usando la biblioteca estándar `socket`.

Esto no sustituye a herramientas como `dig`, pero permite experimentar dentro del propio cuaderno.


In [ ]:
import socket

dominio = "www.ejemplo.com"

try:
    print(f"Resolviendo {dominio}...")
    resultado = socket.gethostbyname_ex(dominio)
    print(resultado)
except socket.gaierror as error:
    print(f"No se pudo resolver el dominio: {error}")


In [ ]:
import socket

ip = "8.8.8.8"

try:
    print(f"Resolución inversa de {ip}...")
    resultado = socket.gethostbyaddr(ip)
    print(resultado)
except socket.herror as error:
    print(f"No se pudo realizar la resolución inversa: {error}")


## 13. Ejercicios prácticos

Realiza las siguientes pruebas y copia debajo de cada una una captura o el resultado obtenido.

### Ejercicio 1. Resolución directa

Consulta la dirección IP de estos dominios:

```text
www.google.com
www.wikipedia.org
iesalonsodemadrigal.centros.educa.jcyl.es
```

Puedes usar `nslookup`, `dig` o `host`.


In [ ]:
# Escribe aquí tus comandos de resolución directa.
# Ejemplo:
# !nslookup www.google.com


### Ejercicio 2. Resolución inversa

Realiza una consulta inversa de las siguientes IP:

```text
8.8.8.8
1.1.1.1
```

Indica qué nombre devuelve cada una.


In [ ]:
# Escribe aquí tus comandos de resolución inversa.
# Ejemplo:
# !nslookup 8.8.8.8


### Ejercicio 3. Consulta a un servidor DNS concreto

Realiza la misma consulta usando dos servidores DNS diferentes:

```text
8.8.8.8
1.1.1.1
```

Compara si la respuesta es igual o diferente.


In [ ]:
# Ejemplo:
# !nslookup www.wikipedia.org 8.8.8.8
# !nslookup www.wikipedia.org 1.1.1.1


### Ejercicio 4. Registro MX

Consulta los registros MX de un dominio.

Ejemplo:

```bash
nslookup -type=mx gmail.com
```

Responde:

- ¿Qué servidores de correo aparecen?
- ¿Qué significa la prioridad del registro MX?


In [ ]:
# Escribe aquí tu consulta de registros MX.
# Ejemplo:
# !nslookup -type=mx gmail.com


### Ejercicio 5. Diagnóstico con `dig +trace`

Si tienes `dig` instalado, ejecuta:

```bash
dig www.wikipedia.org +trace
```

Responde:

- ¿Por qué aparecen varios niveles de servidores?
- ¿En qué parte de la salida se observan los servidores raíz?
- ¿En qué parte aparecen servidores autoritativos?


In [ ]:
# Escribe aquí tu consulta con dig +trace.
# !dig www.wikipedia.org +trace


## 14. Preguntas de repaso

1. ¿Qué problema resuelve DNS?
2. ¿Qué diferencia hay entre un servidor DNS recursivo y uno autoritativo?
3. ¿Qué es una zona DNS?
4. ¿Qué diferencia hay entre AXFR e IXFR?
5. ¿Para qué sirve un registro MX?
6. ¿Qué tipo de registro se usa para resolución inversa?
7. ¿Qué es DDNS y cuándo puede ser útil?
8. ¿Qué ventaja tiene usar reenviadores DNS?
9. ¿Por qué una transferencia de zona abierta puede ser un riesgo de seguridad?
10. ¿Qué comando usarías para ver el proceso completo de resolución DNS?


## 15. Resumen final

El DNS permite traducir nombres de dominio a direcciones IP y es imprescindible para el funcionamiento de Internet y de muchas redes locales.

Sus ideas clave son:

- El DNS es jerárquico y descentralizado.
- La resolución de nombres puede implicar caché local, servidores recursivos, raíz, TLD y autoritativos.
- Las zonas DNS permiten administrar partes del espacio de nombres.
- Las transferencias de zona mantienen sincronizados servidores primarios y secundarios.
- Los registros DNS definen servicios, alias, servidores de correo, servidores autoritativos y resolución inversa.
- Herramientas como `nslookup`, `dig`, `host` y `ping` permiten diagnosticar problemas de resolución.
